# VOIS × AICTE Batch 1 (2026–2027)
# Major Project: Seasonal Agriculture Performance Analysis

**Project Type:** Data Analytics / Exploratory & Statistical Analysis  
**Dataset:** `seasonal_agriculture_performance_dataset.csv`  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, SciPy, Jupyter/Google Colab

---

## 1. Project Overview

Agricultural performance is affected by seasonal environmental conditions, farming practices, resource usage, crop selection and market conditions. This project analyzes the provided agricultural dataset to understand **how agricultural performance changes across Kharif, Rabi and Zaid seasons**.

The analysis is intentionally focused on **seasonal differences**, rather than being a generic dataset exploration.

### Primary objective
Identify meaningful seasonal patterns, differences, relationships, unusual observations and evidence-based recommendations that can support better seasonal agricultural planning.

### Key analytical themes
- Seasonal yield and production performance
- Revenue, cost and profitability by season
- Environmental conditions across seasons
- Resource and water usage
- Crop and irrigation differences within seasons
- Regional consistency and variation
- Statistical significance of seasonal differences
- Outlier and unusual-pattern investigation
- Evidence-based recommendations and limitations


## 2. Business / Agricultural Questions

The project addresses the following questions:

1. How does agricultural performance vary across seasons?
2. Which season has the strongest yield, production and economic performance?
3. How do rainfall, temperature, humidity, soil moisture and sunlight differ by season?
4. Does resource usage change across seasons?
5. How do irrigation methods perform across seasons?
6. Are seasonal differences consistent across crops and regions?
7. Are environmental variables associated with yield?
8. Are there statistically significant differences in yield and profit between seasons?
9. What unusual or potentially influential observations exist?
10. What practical recommendations can reasonably be made from the dataset?


## 3. Analytical Approach

**Workflow**

`Data Understanding → Data Quality → Cleaning → Feature Engineering → EDA → Seasonal Comparison → Crop/Region Analysis → Resource Analysis → Statistical Testing → Anomaly Analysis → Insights → Recommendations`

### Important analytical principles
- Use **mean and median together**, because financial variables and agricultural yield can be skewed.
- Avoid treating correlation as causation.
- Compare groups with appropriate denominators and sample sizes.
- Use statistical tests to support—not replace—business interpretation.
- Clearly distinguish observations supported by the dataset from assumptions that require field validation.


In [ ]:
# 4. Import libraries and configure the notebook

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid", context="notebook")
print("Libraries loaded successfully.")


## 5. Load the Dataset

The notebook first checks whether the CSV is already available in the Colab working directory. If it is not found, a file-upload control is provided.

This makes the notebook reusable for the VOIS/AICTE project submission.


In [ ]:
# Locate the dataset

DATASET_NAME = "seasonal_agriculture_performance_dataset.csv"
candidate_paths = [
    f"/content/{DATASET_NAME}",
    f"/mnt/data/{DATASET_NAME}",
    DATASET_NAME
]

DATA_PATH = next((p for p in candidate_paths if os.path.exists(p)), None)

if DATA_PATH is None:
    from google.colab import files
    print("Dataset not found in the current environment. Please upload the CSV file.")
    uploaded = files.upload()
    DATA_PATH = next(iter(uploaded.keys()))

df_raw = pd.read_csv(DATA_PATH)

print(f"Dataset loaded from: {DATA_PATH}")
print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]:,}")
df_raw.head()


In [ ]:
# Basic structural inspection

print("Column names:")
print(df_raw.columns.tolist())

print("\nData types:")
display(df_raw.dtypes.to_frame("dtype"))

print("\nDataset dimensions:", df_raw.shape)


In [ ]:
# Unique values in important categorical variables

categorical_cols = df_raw.select_dtypes(include="object").columns.tolist()

for col in categorical_cols:
    print(f"\n{col} | unique values = {df_raw[col].nunique()}")
    print(df_raw[col].value_counts(dropna=False).head(15))


## 6. Data Quality Assessment

We inspect:
- Missing values
- Duplicate rows
- Duplicate farm IDs
- Invalid numeric values
- Category consistency
- Basic distribution ranges

Missing values are not automatically dropped. Where appropriate, numeric missing values are handled using **median imputation**, which is less sensitive to skew and extreme values than mean imputation.


In [ ]:
# Missing values and duplicates

quality_summary = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_pct": df_raw.isna().mean() * 100,
    "dtype": df_raw.dtypes.astype(str)
}).sort_values(["missing_count", "missing_pct"], ascending=False)

display(quality_summary)

print("Exact duplicate rows:", df_raw.duplicated().sum())
if "Farm_ID" in df_raw.columns:
    print("Duplicate Farm_ID values:", df_raw["Farm_ID"].duplicated().sum())


In [ ]:
# Check numeric ranges and potential invalid values

numeric_cols = df_raw.select_dtypes(include=np.number).columns.tolist()

range_checks = pd.DataFrame({
    "min": df_raw[numeric_cols].min(),
    "max": df_raw[numeric_cols].max(),
    "mean": df_raw[numeric_cols].mean(),
    "median": df_raw[numeric_cols].median()
})

display(range_checks)


In [ ]:
# Data cleaning

df = df_raw.copy()

# Remove exact duplicate rows
df = df.drop_duplicates().copy()

# Standardize text columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

# Convert numeric columns explicitly where applicable
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Median-impute missing numeric observations
missing_before = df[numeric_cols].isna().sum()
for col in numeric_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

print("Missing numeric values before imputation:")
display(missing_before[missing_before > 0].to_frame("missing_count"))

print("Remaining missing values:", int(df.isna().sum().sum()))
print("Rows after cleaning:", len(df))


### Cleaning decision

The dataset contains a small number of missing observations in selected continuous variables. Since the project is primarily comparative and exploratory, median imputation preserves the available records while limiting the influence of extreme values.

**Important:** imputed values should not be interpreted as observed field measurements. A production agricultural system would ideally use source-level validation or domain-specific imputation rules.


## 7. Feature Engineering

To compare farms fairly across different farm sizes, the analysis emphasizes **per-hectare** and **efficiency** measures where appropriate.

New measures:
- Revenue per hectare
- Cost per hectare
- Profit per hectare
- Water used per hectare
- Profit margin
- Revenue-to-cost ratio
- Season × Crop grouping


In [ ]:
# Feature engineering

df["Revenue_per_Hectare_INR"] = df["Revenue_INR"] / df["Farm_Area_Hectares"]
df["Cost_per_Hectare_INR"] = df["Total_Cost_INR"] / df["Farm_Area_Hectares"]
df["Profit_per_Hectare_INR"] = df["Profit_INR"] / df["Farm_Area_Hectares"]
df["Water_per_Hectare_m3"] = df["Water_Used_m3"] / df["Farm_Area_Hectares"]
df["Profit_Margin_pct"] = np.where(
    df["Revenue_INR"] != 0,
    df["Profit_INR"] / df["Revenue_INR"] * 100,
    np.nan
)
df["Revenue_to_Cost_Ratio"] = np.where(
    df["Total_Cost_INR"] != 0,
    df["Revenue_INR"] / df["Total_Cost_INR"],
    np.nan
)

print("Engineered columns added:")
print([
    "Revenue_per_Hectare_INR",
    "Cost_per_Hectare_INR",
    "Profit_per_Hectare_INR",
    "Water_per_Hectare_m3",
    "Profit_Margin_pct",
    "Revenue_to_Cost_Ratio"
])

display(df.head())


# 8. Descriptive Statistics

The first seasonal comparison uses both **mean** and **median** values. This is especially important for variables such as yield, revenue and profit where extreme values can materially affect the mean.


In [ ]:
season_order = [s for s in ["Kharif", "Rabi", "Zaid"] if s in df["Season"].unique()]

season_metrics = [
    "Yield_Tonnes_Ha",
    "Production_Tonnes",
    "Revenue_INR",
    "Total_Cost_INR",
    "Profit_INR",
    "Water_Used_m3",
    "Water_Efficiency_t_per_1000m3",
    "Disease_Pest_Risk_pct"
]

season_summary = df.groupby("Season")[season_metrics].agg(["mean", "median"]).reindex(season_order)
display(season_summary.round(2))


## 9. Seasonal Performance Comparison

### 9.1 Yield by Season
Yield is the most direct production-performance measure because it is normalized by cultivated area.

We compare:
- Average yield
- Median yield
- Distribution and spread
- Potential outliers


In [ ]:
# Yield distribution by season

plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="Season", y="Yield_Tonnes_Ha", order=season_order)
plt.title("Yield Distribution Across Seasons")
plt.xlabel("Season")
plt.ylabel("Yield (Tonnes/Ha)")
plt.tight_layout()
plt.show()


In [ ]:
# Mean vs median yield

yield_season = df.groupby("Season")["Yield_Tonnes_Ha"].agg(["mean", "median", "std", "count"]).reindex(season_order)
display(yield_season.round(2))

plt.figure(figsize=(9, 5))
plot_df = yield_season[["mean", "median"]].reset_index().melt(
    id_vars="Season", var_name="Statistic", value_name="Yield"
)
sns.barplot(data=plot_df, x="Season", y="Yield", hue="Statistic", order=season_order)
plt.title("Mean vs Median Yield by Season")
plt.ylabel("Yield (Tonnes/Ha)")
plt.tight_layout()
plt.show()


### 9.2 Production and Farm Size

Total production is influenced by both yield and cultivated area. Therefore, total production should not be interpreted as pure agricultural productivity.


In [ ]:
# Production and farm area by season

prod_area = df.groupby("Season").agg(
    Avg_Production_Tonnes=("Production_Tonnes", "mean"),
    Median_Production_Tonnes=("Production_Tonnes", "median"),
    Avg_Farm_Area_Hectares=("Farm_Area_Hectares", "mean"),
    Farms=("Farm_ID", "count")
).reindex(season_order)

display(prod_area.round(2))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(prod_area.index))
ax.bar(x, prod_area["Avg_Production_Tonnes"].values)
ax.set_xticks(x)
ax.set_xticklabels(prod_area.index)
ax.set_title("Average Production by Season")
ax.set_xlabel("Season")
ax.set_ylabel("Production (Tonnes)")
plt.tight_layout()
plt.show()


## 10. Economic Performance Across Seasons

Economic performance is examined using:
- Revenue
- Total cost
- Profit
- Profit per hectare
- Profit margin

Per-hectare metrics help control for differences in farm area.


In [ ]:
economic_summary = df.groupby("Season").agg(
    Avg_Revenue_INR=("Revenue_INR", "mean"),
    Median_Revenue_INR=("Revenue_INR", "median"),
    Avg_Cost_INR=("Total_Cost_INR", "mean"),
    Avg_Profit_INR=("Profit_INR", "mean"),
    Median_Profit_INR=("Profit_INR", "median"),
    Avg_Profit_per_Hectare=("Profit_per_Hectare_INR", "mean"),
    Median_Profit_per_Hectare=("Profit_per_Hectare_INR", "median"),
    Avg_Profit_Margin_pct=("Profit_Margin_pct", "mean")
).reindex(season_order)

display(economic_summary.round(2))


In [ ]:
# Profit comparison

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x="Season", y="Profit_per_Hectare_INR", order=season_order, ax=ax)
ax.set_title("Profit per Hectare Distribution Across Seasons")
ax.set_xlabel("Season")
ax.set_ylabel("Profit per Hectare (INR)")
plt.tight_layout()
plt.show()


## 11. Environmental Conditions by Season

The dataset contains several environmental variables that may differ across seasons:

- Rainfall
- Average temperature
- Humidity
- Sunlight
- Soil moisture
- Soil pH

These are examined as **observational seasonal differences**, not proof of causal effects.


In [ ]:
environment_cols = [
    "Rainfall_mm",
    "Avg_Temperature_C",
    "Humidity_pct",
    "Sunlight_Hours_Day",
    "Soil_Moisture_pct",
    "Soil_pH"
]

environment_summary = df.groupby("Season")[environment_cols].mean().reindex(season_order)
display(environment_summary.round(2))


In [ ]:
# Standardized seasonal environmental profile

env_z = environment_summary.copy()
env_z = (env_z - env_z.mean()) / env_z.std()

plt.figure(figsize=(11, 6))
sns.heatmap(env_z.T, annot=True, fmt=".2f", center=0)
plt.title("Relative Environmental Profile by Season (Z-scores)")
plt.xlabel("Season")
plt.ylabel("Environmental Variable")
plt.tight_layout()
plt.show()


## 12. Resource Usage and Irrigation

Resource usage is evaluated through:
- Fertilizer
- Pesticide
- Nitrogen, phosphorus and potassium
- Water use
- Water efficiency
- Irrigation method

A higher water-efficiency value indicates more tonnes of output per 1,000 m³ of water, according to the dataset's definition.


In [ ]:
resource_cols = [
    "Nitrogen_kg_ha", "Phosphorus_kg_ha", "Potassium_kg_ha",
    "Fertilizer_kg_ha", "Pesticide_Litre_ha",
    "Water_Used_m3", "Water_Efficiency_t_per_1000m3"
]

resource_summary = df.groupby("Season")[resource_cols].mean().reindex(season_order)
display(resource_summary.round(2))


In [ ]:
# Irrigation method × season: yield

irrigation_yield = pd.pivot_table(
    df,
    index="Irrigation_Method",
    columns="Season",
    values="Yield_Tonnes_Ha",
    aggfunc="mean"
).reindex(columns=season_order)

display(irrigation_yield.round(2))

plt.figure(figsize=(10, 6))
sns.heatmap(irrigation_yield, annot=True, fmt=".2f")
plt.title("Average Yield by Irrigation Method and Season")
plt.xlabel("Season")
plt.ylabel("Irrigation Method")
plt.tight_layout()
plt.show()


In [ ]:
# Irrigation method × season: water efficiency

irrigation_eff = pd.pivot_table(
    df,
    index="Irrigation_Method",
    columns="Season",
    values="Water_Efficiency_t_per_1000m3",
    aggfunc="mean"
).reindex(columns=season_order)

display(irrigation_eff.round(2))

plt.figure(figsize=(10, 6))
sns.heatmap(irrigation_eff, annot=True, fmt=".2f")
plt.title("Water Efficiency by Irrigation Method and Season")
plt.xlabel("Season")
plt.ylabel("Irrigation Method")
plt.tight_layout()
plt.show()


## 13. Crop × Season Analysis

Seasonal performance can be misleading if the crop mix changes between seasons. We therefore examine crop-level performance within each season.

This helps answer:

> Is a seasonal difference still visible when comparing crops rather than only comparing all farms together?


In [ ]:
crop_season_yield = pd.pivot_table(
    df,
    index="Crop",
    columns="Season",
    values="Yield_Tonnes_Ha",
    aggfunc="mean"
).reindex(columns=season_order)

display(crop_season_yield.round(2))

plt.figure(figsize=(11, 7))
sns.heatmap(crop_season_yield, annot=True, fmt=".2f")
plt.title("Average Yield by Crop and Season")
plt.xlabel("Season")
plt.ylabel("Crop")
plt.tight_layout()
plt.show()


In [ ]:
# Profit per hectare by crop and season

crop_season_profit = pd.pivot_table(
    df,
    index="Crop",
    columns="Season",
    values="Profit_per_Hectare_INR",
    aggfunc="mean"
).reindex(columns=season_order)

display(crop_season_profit.round(0))

plt.figure(figsize=(11, 7))
sns.heatmap(crop_season_profit, annot=True, fmt=".0f", center=0)
plt.title("Average Profit per Hectare by Crop and Season")
plt.xlabel("Season")
plt.ylabel("Crop")
plt.tight_layout()
plt.show()


## 14. Regional Analysis

Seasonal conclusions should be checked across geography. We compare:
- State
- District
- Season

The goal is to identify whether a pattern is broadly distributed or concentrated in particular locations.


In [ ]:
state_season_yield = pd.pivot_table(
    df,
    index="State",
    columns="Season",
    values="Yield_Tonnes_Ha",
    aggfunc="mean"
).reindex(columns=season_order)

display(state_season_yield.round(2))

plt.figure(figsize=(12, 7))
sns.heatmap(state_season_yield, annot=True, fmt=".2f")
plt.title("Average Yield by State and Season")
plt.xlabel("Season")
plt.ylabel("State")
plt.tight_layout()
plt.show()


In [ ]:
# Best and weakest season by state

state_best = state_season_yield.idxmax(axis=1).rename("Best_Season")
state_worst = state_season_yield.idxmin(axis=1).rename("Lowest_Season")

state_comparison = pd.concat([state_season_yield, state_best, state_worst], axis=1)
display(state_comparison)


## 15. Relationships Between Environment, Resources and Yield

Correlation measures linear association. It does **not** establish that one variable causes another.

We examine Pearson correlations between yield and selected environmental/resource/economic variables.


In [ ]:
relationship_cols = [
    "Yield_Tonnes_Ha",
    "Rainfall_mm",
    "Avg_Temperature_C",
    "Humidity_pct",
    "Sunlight_Hours_Day",
    "Soil_pH",
    "Soil_Moisture_pct",
    "Nitrogen_kg_ha",
    "Phosphorus_kg_ha",
    "Potassium_kg_ha",
    "Fertilizer_kg_ha",
    "Pesticide_Litre_ha",
    "Seed_Quality_Score",
    "Water_Used_m3",
    "Water_Efficiency_t_per_1000m3",
    "Disease_Pest_Risk_pct"
]

corr_matrix = df[relationship_cols].corr(method="pearson")

yield_corr = corr_matrix["Yield_Tonnes_Ha"].sort_values(ascending=False)
display(yield_corr.to_frame("Pearson_Correlation_with_Yield"))


In [ ]:
# Correlation heatmap for key variables

key_corr_cols = [
    "Yield_Tonnes_Ha",
    "Rainfall_mm",
    "Avg_Temperature_C",
    "Humidity_pct",
    "Soil_Moisture_pct",
    "Seed_Quality_Score",
    "Water_Used_m3",
    "Water_Efficiency_t_per_1000m3",
    "Disease_Pest_Risk_pct"
]

plt.figure(figsize=(11, 8))
sns.heatmap(df[key_corr_cols].corr(), annot=True, fmt=".2f", center=0, cmap="coolwarm")
plt.title("Correlation Matrix: Yield and Selected Factors")
plt.tight_layout()
plt.show()


## 16. Statistical Testing

Visual differences are useful, but statistical testing helps assess whether observed seasonal differences are likely to be more than random variation.

### Test 1: One-way ANOVA
**H₀:** Mean yield is equal across seasons.  
**H₁:** At least one seasonal mean differs.

### Test 2: Kruskal–Wallis
A non-parametric alternative that compares the distributions/ranks across the three seasons and is useful when normality assumptions are questionable.

### Interpretation
A p-value below 0.05 is treated as evidence against the null hypothesis. Statistical significance does not automatically mean the difference is practically important.


In [ ]:
# ANOVA and Kruskal-Wallis tests for yield

groups_yield = [
    df.loc[df["Season"] == s, "Yield_Tonnes_Ha"].dropna()
    for s in season_order
]

anova_stat, anova_p = stats.f_oneway(*groups_yield)
kw_stat, kw_p = stats.kruskal(*groups_yield)

test_results = pd.DataFrame({
    "Test": ["One-way ANOVA", "Kruskal-Wallis"],
    "Statistic": [anova_stat, kw_stat],
    "p_value": [anova_p, kw_p],
    "Conclusion_at_5pct": [
        "Significant seasonal difference" if anova_p < 0.05 else "Not statistically significant",
        "Significant seasonal difference" if kw_p < 0.05 else "Not statistically significant"
    ]
})

display(test_results)


### Pairwise seasonal comparison

When an overall test indicates a difference, pairwise comparisons help identify which seasons differ.

We use the **Mann–Whitney U test** with Bonferroni adjustment for the three pairwise comparisons. This provides a conservative non-parametric comparison.


In [ ]:
# Pairwise Mann-Whitney U tests with Bonferroni correction

from itertools import combinations

pairwise_results = []
n_comparisons = len(list(combinations(season_order, 2)))

for a, b in combinations(season_order, 2):
    x = df.loc[df["Season"] == a, "Yield_Tonnes_Ha"].dropna()
    y = df.loc[df["Season"] == b, "Yield_Tonnes_Ha"].dropna()
    u_stat, p = stats.mannwhitneyu(x, y, alternative="two-sided")
    p_adj = min(p * n_comparisons, 1.0)
    pairwise_results.append({
        "Season_A": a,
        "Season_B": b,
        "U_statistic": u_stat,
        "Raw_p_value": p,
        "Bonferroni_adjusted_p": p_adj,
        "Significant_at_5pct": p_adj < 0.05
    })

pairwise_df = pd.DataFrame(pairwise_results)
display(pairwise_df)


## 17. Effect Size: Practical Importance

With thousands of observations, even small differences can become statistically significant. Therefore, we also calculate an effect size for the overall seasonal yield difference.

**Eta-squared (η²)** estimates the proportion of variance in yield associated with season in a one-way ANOVA framework.

This should be interpreted as an association measure, not a causal effect.


In [ ]:
# Eta-squared effect size for season and yield

grand_mean = df["Yield_Tonnes_Ha"].mean()

between_ss = sum(
    len(g) * (g.mean() - grand_mean) ** 2
    for g in groups_yield
)

total_ss = ((df["Yield_Tonnes_Ha"] - grand_mean) ** 2).sum()

eta_squared = between_ss / total_ss if total_ss != 0 else np.nan

print(f"Eta-squared (η²): {eta_squared:.4f}")

if eta_squared < 0.01:
    interpretation = "very small association"
elif eta_squared < 0.06:
    interpretation = "small association"
elif eta_squared < 0.14:
    interpretation = "moderate association"
else:
    interpretation = "relatively large association"

print("Interpretation:", interpretation)


## 18. Outlier and Unusual-Pattern Analysis

Agricultural datasets can contain extreme observations because of:
- unusually productive farms,
- crop-specific production characteristics,
- very large farm areas,
- exceptional prices,
- data-entry issues,
- unusual environmental conditions.

Outliers are **not automatically removed**. First, they are investigated to determine whether they are plausible observations.


In [ ]:
# IQR-based outlier counts for selected performance variables

outlier_summary = []

for col in ["Yield_Tonnes_Ha", "Production_Tonnes", "Revenue_INR", "Profit_INR", "Water_Used_m3"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (df[col] < lower) | (df[col] > upper)

    outlier_summary.append({
        "Variable": col,
        "Q1": q1,
        "Q3": q3,
        "Lower_Bound": lower,
        "Upper_Bound": upper,
        "Outlier_Count": int(mask.sum()),
        "Outlier_Percent": mask.mean() * 100
    })

outlier_df = pd.DataFrame(outlier_summary)
display(outlier_df.round(2))


In [ ]:
# Inspect the most extreme yield observations

extreme_yield = df.nlargest(15, "Yield_Tonnes_Ha")[
    [
        "Farm_ID", "State", "District", "Crop", "Season",
        "Farm_Area_Hectares", "Yield_Tonnes_Ha",
        "Production_Tonnes", "Revenue_INR", "Profit_INR",
        "Water_Efficiency_t_per_1000m3"
    ]
]

display(extreme_yield)


### Important observation

Extreme values should be checked against the source data before being deleted. If a value is valid and represents a real agricultural condition or crop characteristic, removing it could distort the analysis.

For this project, the main seasonal comparisons therefore show **distribution plots and medians alongside means**, reducing the risk of relying on a single average.


## 19. Season × Crop Mix

A major source of apparent seasonal differences can be **composition**: if some crops occur more frequently in one season than another, aggregate performance may partly reflect crop mix.

We therefore calculate crop proportions by season.


In [ ]:
crop_mix = pd.crosstab(
    df["Season"], df["Crop"], normalize="index"
).reindex(season_order) * 100

display(crop_mix.round(2))

plt.figure(figsize=(12, 6))
crop_mix.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.title("Crop Composition Within Each Season")
plt.xlabel("Season")
plt.ylabel("Share of Farms (%)")
plt.legend(title="Crop", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 20. Additional Robustness Check: Median-Based Seasonal Ranking

Because the dataset contains some extreme values, we compare season rankings using:
- mean yield,
- median yield,
- mean profit per hectare,
- median profit per hectare,
- mean water efficiency.

If the same season remains near the top under multiple measures, the conclusion is more robust.


In [ ]:
robustness = df.groupby("Season").agg(
    Mean_Yield=("Yield_Tonnes_Ha", "mean"),
    Median_Yield=("Yield_Tonnes_Ha", "median"),
    Mean_Profit_per_Ha=("Profit_per_Hectare_INR", "mean"),
    Median_Profit_per_Ha=("Profit_per_Hectare_INR", "median"),
    Mean_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean")
).reindex(season_order)

display(robustness.round(2))

ranking = robustness.rank(ascending=False, method="min")
display(ranking.astype(int).add_suffix("_Rank"))


# 21. Automated Key Findings

The following cells convert the analysis into concise evidence-based findings. This avoids hard-coding conclusions and keeps the notebook reusable if the dataset is updated.


In [ ]:
# Generate core findings from the actual dataset

season_yield_mean = df.groupby("Season")["Yield_Tonnes_Ha"].mean().reindex(season_order)
season_yield_median = df.groupby("Season")["Yield_Tonnes_Ha"].median().reindex(season_order)
season_profit_median = df.groupby("Season")["Profit_per_Hectare_INR"].median().reindex(season_order)
season_water_eff = df.groupby("Season")["Water_Efficiency_t_per_1000m3"].mean().reindex(season_order)
season_risk = df.groupby("Season")["Disease_Pest_Risk_pct"].mean().reindex(season_order)

best_yield_season = season_yield_mean.idxmax()
worst_yield_season = season_yield_mean.idxmin()
best_profit_season = season_profit_median.idxmax()
best_water_season = season_water_eff.idxmax()
lowest_risk_season = season_risk.idxmin()

print("KEY FINDINGS")
print("-" * 70)
print(f"1. Highest mean yield: {best_yield_season} ({season_yield_mean[best_yield_season]:.2f} tonnes/ha).")
print(f"2. Lowest mean yield: {worst_yield_season} ({season_yield_mean[worst_yield_season]:.2f} tonnes/ha).")
print(f"3. Highest median profit per hectare: {best_profit_season} (INR {season_profit_median[best_profit_season]:,.0f}/ha).")
print(f"4. Highest mean water efficiency: {best_water_season} ({season_water_eff[best_water_season]:.2f} t/1,000 m³).")
print(f"5. Lowest average disease/pest risk: {lowest_risk_season} ({season_risk[lowest_risk_season]:.2f}%).")
print(f"6. ANOVA p-value for yield: {anova_p:.4g}.")
print(f"7. Kruskal-Wallis p-value for yield: {kw_p:.4g}.")
print(f"8. Eta-squared for season → yield association: {eta_squared:.4f} ({interpretation}).")


In [ ]:
# Identify strongest positive/negative linear associations with yield

corr_without_self = yield_corr.drop("Yield_Tonnes_Ha")
strongest_positive = corr_without_self.idxmax()
strongest_negative = corr_without_self.idxmin()

print(f"Strongest positive Pearson association with yield among selected variables: "
      f"{strongest_positive} (r = {corr_without_self[strongest_positive]:.3f})")
print(f"Strongest negative Pearson association with yield among selected variables: "
      f"{strongest_negative} (r = {corr_without_self[strongest_negative]:.3f})")


# 22. Final Findings and Interpretation

Based on the complete analysis, the final interpretation should be structured around the following evidence:

### A. Seasonal performance
Use the mean and median yield tables and boxplots to explain which seasons perform better and whether the ranking is stable.

### B. Economic performance
Use profit per hectare and profit-margin results rather than total profit alone, because farm sizes differ.

### C. Environmental differences
Explain how rainfall, temperature, humidity, sunlight and soil moisture differ between seasons. Avoid claiming that these variables *caused* the yield difference.

### D. Resource use
Compare irrigation, fertilizer and water-efficiency patterns. Highlight cases where higher output is achieved with relatively efficient water use.

### E. Crop and regional consistency
Use the crop × season and state × season tables to determine whether the overall seasonal pattern is widespread or driven by a subset of crops/locations.

### F. Statistical evidence
Report the ANOVA and Kruskal–Wallis results, pairwise comparisons and effect size. A statistically significant result means the observed seasonal distributions differ; it does not prove a causal seasonal mechanism.

### G. Outliers
Document unusually high/low observations and recommend source validation before exclusion.


# 23. Evidence-Based Recommendations

The recommendations below should be finalized using the actual outputs above:

1. **Use season-specific planning:** Align crop and input planning with the season showing the strongest observed performance for each crop/region.
2. **Prefer crop-specific recommendations:** Avoid applying one seasonal rule to every crop because crop × season results may differ.
3. **Prioritize efficient irrigation:** Where the data show strong yield and water-efficiency performance, investigate those irrigation practices for broader adoption.
4. **Use per-hectare economics:** Compare profit per hectare and profit margin when evaluating seasonal profitability.
5. **Strengthen environmental monitoring:** Track rainfall, soil moisture, temperature and humidity because these variables show measurable seasonal variation.
6. **Target disease/pest management:** Higher-risk seasons or locations can be prioritized for monitoring and preventive interventions.
7. **Validate extreme observations:** Investigate unusually high yield, revenue or water-efficiency records against original farm records before using them for policy or planning.
8. **Run deeper causal studies before major decisions:** This dataset is observational. Controlled studies or longitudinal models would be needed to establish causal effects.


# 24. Project Limitations

1. **Observational data:** Associations do not establish causality.
2. **Potential crop-mix effects:** Aggregate seasonal performance can reflect changes in crop composition.
3. **Missing values:** Some numeric observations required median imputation.
4. **Outliers:** Extreme observations may materially influence averages.
5. **Unobserved factors:** Labour availability, farm management quality, pests, input timing, market timing and other variables may affect outcomes.
6. **No time-series structure is explicitly modeled:** The analysis compares seasons but does not establish multi-year trends.
7. **Economic interpretation:** Profit depends on the cost and price structure represented in the dataset; external market validation would be required.
8. **Generalization:** Results apply to the provided dataset and should not automatically be generalized to all agricultural regions.


# 25. Conclusion

This project provides a structured analysis of seasonal agricultural performance using descriptive statistics, visual analytics, crop/region comparisons, resource-efficiency analysis, correlation, statistical testing and outlier investigation.

The central conclusion should be based on **multiple pieces of evidence rather than a single metric**. Seasonal differences in yield, economics, environmental conditions, resource use and risk should be interpreted together, while accounting for crop mix, regional variation and extreme observations.

The analysis can support better seasonal planning by identifying where performance differs, which groups deserve further investigation and which resource-use patterns appear promising. However, field validation and longitudinal or causal analysis would be required before turning these observations into universal agricultural policy.


# 26. Submission Checklist

Before submitting the notebook:

- [x] Dataset loaded and documented
- [x] Data types and structure inspected
- [x] Missing values assessed and handled
- [x] Duplicate records checked
- [x] Relevant features engineered
- [x] Seasonal performance compared
- [x] Economic outcomes analyzed
- [x] Environmental conditions compared
- [x] Resource and irrigation usage analyzed
- [x] Crop × season patterns examined
- [x] State × season patterns examined
- [x] Correlations calculated
- [x] Statistical tests performed
- [x] Effect size calculated
- [x] Outliers investigated
- [x] Evidence-based recommendations provided
- [x] Limitations documented
- [x] Final conclusion included

### Suggested project title for submission

**“Seasonal Agriculture Performance Analysis: Data-Driven Assessment of Yield, Resources, Environment and Economic Outcomes”**
